This programm initializes the calibration for all chosen basins.
- create output folders
- create settingsfiles
- run batch jobs for initialization

In [1]:
# Python-Module für System- und Dateimanagement
import os  # Betriebssystem-Interaktionen
import sys  # Systembezogene Funktionen
from sys import platform  # Erkennen des Betriebssystems
import shutil  # Datei- und Verzeichnisoperationen
import glob  # Suchen von Dateien mit Muster
import stat  # Datei- und Verzeichnisrechte
import time  # Zeitmessung und -steuerung
import datetime  # Arbeiten mit Datum und Uhrzeit
import multiprocessing  # Parallele Verarbeitung
import subprocess  # Ausführen externer Prozesse
from subprocess import Popen, PIPE  # Alternative Möglichkeit für Subprozess-Kommunikation
import pickle  # Speichern und Laden von Python-Objekten (Serialisierung)
import ast  # Abstrakte Syntaxbäume (z. B. Umwandlung von Strings in Python-Objekte)
import configparser  # Einlesen und Schreiben von Konfigurationsdateien
from configparser import ConfigParser  # Alternative Schreibweise

# Numerische Berechnungen & Zufallszahlen
import array  # Arbeit mit Arrays
import random  # Zufallszahlengenerierung
import numpy as np  # Wissenschaftliches Rechnen mit Arrays
from collections import defaultdict, Counter

# Datenverarbeitung & Analyse 
import csv  # Lesen und Schreiben von CSV-Dateien
import xarray as xr  # Arbeiten mit multidimensionalen Arrays (z. B. für NetCDF-Daten)
import cftime  # Arbeiten mit Klimazeitdaten (z. B. NetCDF-Dateien)
import pandas as pd  # Datenanalyse und Tabellenverarbeitung

# Visualisierung
import matplotlib.pyplot as plt  # Erstellen von Diagrammen
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches  

# Geografische Daten & Karten
import cartopy.crs as ccrs  # Koordinatenreferenzsysteme
import cartopy.feature as cfeature  # Geografische Features (z. B. Küstenlinien, Ländergrenzen)
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER  # Formatieren von Kartenrastern
import geopandas as gpd  # Verarbeiten von Geodaten, insbesondere Shapefiles
from shapely.geometry import Point  # Arbeiten mit geometrischen Objekten
from osgeo import gdal  # Geodatenformate lesen/schreiben (z. B. Rasterdaten)



# Data paths and names

#### Data path for initialisation 

In [3]:
folder_init_bash = 'bashFiles'  
folder_init_settings = 'settingsFiles'  
folder_init_init = 'initFiles'  
folder_init_output = 'outputFiles'   

#### Data path for GRDC file

In [4]:
folder_GRDC_data = '../Step0_preparationGRDCFiles/GRDC_data/GRDC_calibrationStations'
GRDC_file_name = 'GRDC_42calibrationStations_FINAL_LatLonFit.nc'

#### Data path for file names

In [5]:
settingsfile_template =  'settings_ccwatm_init_template.ini'  
settingsfile = 'settings_ccwatm_init_' 
SLURM_runInit  = 'SLURM_to_run_ccwatm_init.sh'

# 1. Set default values for calibration parameters

In [6]:
# Set default values for calibration parameters (taken from an original settings-files (Status: Jan 2024))
calibration_parameters_dict = {
    'arnoBeta_add':  0.39,
    'alphaDepl': 0.7,
    'recessionCoeff_factor':  5.278,
    'manningsN':  1.86,
    'normalStorageLimit': 0.44,
    'lakeAFactor': 0.33, 
    'lakeEvaFactor': 1.52 }

# Create a list from the calibration parameter keys 
calibration_parameters_list = list(calibration_parameters_dict.keys())

# 2. Define and control the final selection of calibration stations

In [7]:
# Read in GRDC-data-file 
xr_obs_mod = xr.open_dataset(folder_GRDC_data + '/' + GRDC_file_name)

# Read in List of station IDs
list_of_station_IDs = xr_obs_mod['id'].values

# This is where the number of used stations could be reduces if necessary
list_of_station_IDs_calibration = list_of_station_IDs

print('Total number of stations in GRDC-file: ' + str(len(list_of_station_IDs)))
print('Total number of selected calibration stations: ' + str(len(list_of_station_IDs_calibration)))

Total number of stations in GRDC-file: 42
Total number of selected calibration stations: 42


##  --- no changes necessary below ---

# 3. Automatically create settings-files for the initialisation process 

#### Automatically delete all entries in DKRZreturn

In [8]:
def empty_folder(folder):
    if not os.path.isdir(folder): return
    for item in os.listdir(folder):
        path = os.path.join(folder, item)
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
    time.sleep(5) 

for f in ["DKRZreturn"]:
    empty_folder(f)

#### Automatically delete all entries in initFiles, outputFiles und settingsFiles

In [9]:
def empty_folder(folder):
    if not os.path.isdir(folder): return
    for item in os.listdir(folder):
        path = os.path.join(folder, item)
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
    time.sleep(5)  
    
for f in ["settingsFiles", "initFiles", "outputFiles"]:
    empty_folder(f)

#### Automatically create output folder for all calibration stations

In [10]:
# Vorab Output-Ordner erstellen
for ID in list_of_station_IDs_calibration: 
    # Befehlszeile für Erstellen des individuellen Output-Ordner
    mkdir_command = 'mkdir -p ' + folder_init_output + '/' + str(int(ID))
    # Befehlszeile ausführen und auf Ergebnis warten
    subprocess.run(mkdir_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

#### Automatically create settings-file for all calibration stations

In [11]:
# Create dictionary for the file-names of the newly created settingsfiles for initialisation
new_setting_file = {}

# Open the appropriate template, read in data and closes template
setting_file_template =  settingsfile_template
f = open(setting_file_template , "r")
template_xml = f.read()
f.close()

# Automatically create settings-files for each station 
for ID in list_of_station_IDs_calibration: 
    # Duplication from the original settings-file template: 
    template_xml_new = template_xml
    # Modifications from the original settings-file template: Platzhalter %out_path', '%init_path' und '%grdc_id' ersetzt
    template_xml_new = template_xml_new.replace('%out_path', folder_init_output + '/' + str(int(ID)))
    template_xml_new = template_xml_new.replace('%init_path', folder_init_init)
    template_xml_new = template_xml_new.replace('%grdc_id', str(int(ID)))
    # Modifikation from the original settings-file template: Gauges
    lon_fit_list = xr_obs_mod.sel(id=ID).gauged_lon.values
    lat_fit_list = xr_obs_mod.sel(id=ID).gauged_lat.values    
    gauges_str = ' ' + str(lon_fit_list) + ' ' + str(lat_fit_list)
    template_xml_new = template_xml_new.replace('%gauges', gauges_str)
    
    # Modification from the original settings-file template: calibration parameters 
    for parameter in calibration_parameters_list:
        template_xml_new = template_xml_new.replace('%'+ parameter, str(calibration_parameters_dict[parameter]))

    # Create a new settingsfile (duplicated and modified from the template!) with the stations ID ending! 
    new_setting_file[ID] = folder_init_settings + '/' + settingsfile + str(int(ID)) + '.ini'    
    f = open(new_setting_file[ID], "w+")
    f.write(template_xml_new)
    f.close() 

# 4. Automatically run those settings-files for each calibration station with the same SLURM-file 

In [12]:
# Path to the slurm-file 'run_cwatm_slurm-shared.sh', which will be used to run the CWatM model
slurmFile = SLURM_runInit

# Automatically run those initialisation files for each station
for ID in list_of_station_IDs_calibration:    
    # Befehlszeile zum Ausführen des SLURM-files
    # Als zusätzliches Argument wird der Name des initialisation files new_setting_file[ID] übergeben. Das von uns geschriebene Bash-file 'SLURM_to_run_cwatm.sh' wartet schon darauf! 
    bash_command = 'sbatch ' + slurmFile + ' ' + new_setting_file[ID]
    # Befehlszeile ausführen und auf Ergebnis warten
    # subprocess.Popen wird verwendet, um den SLURM-Job im Hintergrund auszuführen, sodass das Jupyter-Notebook nicht blockiert wird und weiterhin ausgeführt werden kann, ohne auf die Fertigstellung des Jobs zu warten.
    # subprocess.Popen is important so that computing can be parallel! Saves a lot of time! 
    result = subprocess.Popen(bash_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    output, error = result.communicate() 
    if result.returncode == 0:
        print(f'For station {str(ID)}: {output.decode()} ')
    else:
        print(f'For station {str(ID)}: Error occured {error.decode()} ')

For station 6113050: Submitted batch job 20992226
 
For station 6116200: Submitted batch job 20992227
 
For station 6122260: Submitted batch job 20992228
 
For station 6123400: Submitted batch job 20992230
 
For station 6125100: Submitted batch job 20992231
 
For station 6136145: Submitted batch job 20992232
 
For station 6140400: Submitted batch job 20992233
 
For station 6142150: Submitted batch job 20992234
 
For station 6172050: Submitted batch job 20992235
 
For station 6212740: Submitted batch job 20992236
 
For station 6226800: Submitted batch job 20992237
 
For station 6227510: Submitted batch job 20992238
 
For station 6232911: Submitted batch job 20992239
 
For station 6233410: Submitted batch job 20992240
 
For station 6233510: Submitted batch job 20992241
 
For station 6243050: Submitted batch job 20992242
 
For station 6335240: Submitted batch job 20992243
 
For station 6337200: Submitted batch job 20992244
 
For station 6340160: Submitted batch job 20992245
 
For station 

# 5. Check status of batch jobs
COMPUTING TIME: Really depends on the ressources of DKRZ. Approximately 10 min computing time per station. Parallel computing of different stations possible. 

In [2]:
command = "squeue -u USERID"
result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
if result.returncode == 0:
    print(result.stdout.decode()) 

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          21003745 interacti spawner-  g300123  R       1:09      1 l40048

